In [ ]:
PRED  (Entity, Pred)
  └── PRED-LIST  (PredList = List{Pred})
        └── ILP-CORE  (BK, POS, NEG : PredList)
              └── METAVAR  (SecondOrderVar, FirstOrderVar)
                    └── METALITERAL-BASE  ([P:A,B])
                          └── METALITERAL  (MetaBody)
                                └── METARULE-BASE  (Metarule)
                                      └── METARULE  (MetaruleList, METARULES)
                                            └── GRANDPARENT-ILP-NEW2  (bài toán cụ thể)
                                                  └── ILP-EXTRACT  (engine đọc cấu trúc)

## MODULE TRÍCH XUẤT SIÊU MẪU (ILP-EXTRACT)
1. Tổng quan hệ thống
Module ILP-EXTRACT.maude đóng vai trò là tầng Hạ tầng Phản chiếu (Reflection Infrastructure).
Trong hệ thống Inductive Logic Programming (ILP), module này chịu trách nhiệm “soi rọi” và phân tích cấu trúc của các Metarules (Siêu luật) và Facts (Sự kiện) ở cấp độ Meta-level.
Nói cách khác, nó cung cấp bộ công cụ phản chiếu giúp máy tính có thể “mổ xẻ” các dòng code logic thành các thành phần dữ liệu có thể tính toán được.

2. Các phân đoạn chức năng cốt lõi
2.1 Quản trị Danh sách Đối số (TermList Utilities)
Do Maude Meta-level không hỗ trợ sẵn các hàm thao tác trên TermList, module này cung cấp thư viện tiện ích cơ bản gồm:
headTL / tailTL — Truy xuất phần tử đầu và phần còn lại của danh sách, phục vụ cho cơ chế đệ quy.

nthTL — Truy cập phần tử tại vị trí N; đây là hàm then chốt để trích xuất chính xác các biến trong một meta-literal.
Ví dụ: Lấy biến B từ vị trí thứ 2 trong [P : A, B].

appendTL — Hợp nhất các danh sách Term, hỗ trợ tái cấu trúc dữ liệu sau xử lý.

2.2 Cơ chế Phản chiếu & Logic “Phẳng hóa” (Flattening)
Các Metarule thường có cấu trúc cây lồng nhau (Nested Tree).
Để thuật toán tìm kiếm hiệu quả, hệ thống cần chuyển chúng về dạng danh sách tuyến tính.

Các hàm chính:

functorOf — Trích xuất định danh (Qid) của hàm hoặc vị từ.
Ví dụ: Từ mother(ann, amy) lấy được 'mother.

flattenGeneric — Hàm đệ quy thực hiện nhiệm vụ “duỗi phẳng” cấu trúc cây của các toán tử _ | _, _,_, và __.

Mục tiêu:
Chuyển đổi [P ∣ [Q ∣ R]] thành danh sách phẳng 'P, 'Q, 'R, giúp ILP engine dễ dàng quản lý và gán giá trị cho các biến vị từ.

2.3 Phân tích Mô hình Liên kết Biến (Sharing Pattern)
Đây là bộ não đánh giá tính hợp lệ (Admissibility) của một luật trước khi tiến hành chứng minh.

sharesArgVar — Kiểm tra sự hiện diện của các biến dùng chung giữa các literal.

bodyBodyShares — Phát hiện “biến cầu nối” (intermediate variables) giữa các literal trong thân luật.

Ví dụ: Trong luật bắc cầu (Chain rule), hàm này xác nhận biến C là mắt xích nối giữa Q và R.
Nếu thiếu kết nối này, luật sẽ bị coi là rời rạc (Disconnected) và bị loại bỏ ngay lập tức, nhằm tối ưu không gian tìm kiếm.

2.4 Cơ chế Quản lý Phép thế (Substitution Engine)
Tầng này quản lý “bộ nhớ” của quá trình so khớp và gán giá trị.

Subst — Danh sách các cặp ràng buộc (Binding) giữa Biến và Giá trị cụ thể.

extendSubst — Hàm xử lý phép gán biến một cách thông minh và nhất quán:

Khởi tạo: Nếu biến chưa có ràng buộc, thiết lập ràng buộc mới.

Nhất quán: Nếu biến đã có giá trị, kiểm tra tính trùng khớp.

Xung đột: Nếu phát hiện mâu thuẫn (ví dụ: biến A đồng thời là ann và bob), hàm trả về emptySubst → kích hoạt cơ chế Backtrack (Quay lui).